# Lab - The Agentic Loop as a Harness

Three tasks that build the agentic loop the way a production harness does: tools that register themselves, failures that become data, and a loop that stops on stop_reason, keeps a trace, and has a turn cap as a safety net.

| Task | What you learn |
| --- | --- |
| 1 | A decorator registers each tool under its name - no hand-kept map to drift |
| 2 | A tool that raises becomes an is_error result, never a crashed loop |
| 3 | Stop on stop_reason, record every turn, cap the turns and say when it fired |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Use the tools to look up orders and to start returns. "
    "Answer in one or two sentences."
)
QUESTION = "I want to return the jacket from order ORD-12345678."

ORDERS = {
    "ORD-12345678": {"item": "jacket", "status": "delivered", "delivered_days_ago": 20},
}

# What Claude reads about each tool.
TOOLS = [
    {"name": "lookup_order",
     "description": "Look up an order by its ID. Returns the item, the status and how "
                    "many days ago it was delivered.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                      "required": ["order_id"]}},
    {"name": "start_return",
     "description": "Start a return for a delivered order that is inside the 30-day "
                    "window. Returns the return label number.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                      "required": ["order_id"]}},
]

---

## Task 1 - Register tools with a decorator

In [ ]:
# ============================================================
# TASK 1 - Register tools with a decorator
# ============================================================
#
# WHAT TO DO
#   Write the tool decorator: it stores a function in
#   TOOL_FUNCTIONS under the function's own name and hands the
#   function back unchanged. Then decorate the two tools.
#
# WHY IT MATTERS
#   The loop runs whichever tool Claude asks for, by name - so
#   your code needs a map from names to functions. Kept by hand,
#   that map drifts from the functions: one gets renamed, one is
#   forgotten. A decorator registers each function where it is
#   defined, so the map is always complete. This is the pattern
#   agent harnesses and MCP libraries use.
#
# WHERE TO SEE IT IN THE LECTURE
#   "a small dictionary maps each tool name to its function".
#
# HOW TO DO IT
#   Three blanks:
#
#       fn.__name__   the key: the function's own name
#       fn            a decorator returns the function itself
#       "RET-2048"    the return label start_return hands out
# ============================================================

TOOL_FUNCTIONS = {}

try:
    # BEGIN SOLUTION
    def tool(fn):
        TOOL_FUNCTIONS[fn.__name__] = fn
        return fn

    @tool
    def lookup_order(order_id):
        return ORDERS.get(order_id, {"error": "order not found"})

    @tool
    def start_return(order_id):
        order = ORDERS[order_id]   # an unknown ID raises KeyError - Task 2 handles that
        return {"return_label": "RET-2048", "order_id": order_id, "item": order["item"]}
    # SCAFFOLD: def tool(fn):
    # SCAFFOLD:     TOOL_FUNCTIONS[...] = fn   # fn.__name__
    # SCAFFOLD:     return ...   # fn - the function itself, unchanged
    # SCAFFOLD:
    # SCAFFOLD: @tool
    # SCAFFOLD: def lookup_order(order_id):
    # SCAFFOLD:     return ORDERS.get(order_id, {"error": "order not found"})
    # SCAFFOLD:
    # SCAFFOLD: @tool
    # SCAFFOLD: def start_return(order_id):
    # SCAFFOLD:     order = ORDERS[order_id]   # an unknown ID raises KeyError - Task 2 handles that
    # SCAFFOLD:     return {"return_label": ..., "order_id": order_id, "item": order["item"]}   # "RET-2048"
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("registered:", list(TOOL_FUNCTIONS))

check("tool_registry", registry=TOOL_FUNCTIONS, tools=TOOLS)

---

## Task 2 - A tool that crashes must not crash the loop

In [ ]:
# ============================================================
# TASK 2 - A tool that crashes must not crash the loop
# ============================================================
#
# WHAT TO DO
#   Finish run_tools(): for every tool_use block, run the
#   matching function. If the function raises, catch it and
#   send back an error result instead. Every block gets exactly
#   one result, with its id.
#
# WHY IT MATTERS
#   Your tools call databases and services, and sooner or later
#   one raises. Without a try, the exception ends the
#   conversation mid-way. With it, the failure becomes data:
#   Claude gets a result flagged is_error and can explain or try
#   something else. And the API still needs a tool_result for
#   every tool_use block, failed or not.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then, for every tool use block in it".
#
# HOW TO DO IT
#   Three blanks:
#
#       Exception   catch whatever the tool raises
#       block.id    the block this result answers
#       failed      True only when the tool raised
# ============================================================

# BEGIN SOLUTION
def run_tools(response):
    results = []
    for block in response.content:
        if block.type != "tool_use":
            continue
        try:
            output, failed = TOOL_FUNCTIONS[block.name](**block.input), False
        except Exception as exc:
            output, failed = {"error": f"{type(exc).__name__}: {exc}"}, True
        print(f"  run {block.name}({block.input}) -> {output}")
        results.append({"type": "tool_result", "tool_use_id": block.id,
                        "content": json.dumps(output), "is_error": failed})
    return results
# SCAFFOLD: def run_tools(response):
# SCAFFOLD:     results = []
# SCAFFOLD:     for block in response.content:
# SCAFFOLD:         if block.type != "tool_use":
# SCAFFOLD:             continue
# SCAFFOLD:         try:
# SCAFFOLD:             output, failed = TOOL_FUNCTIONS[block.name](**block.input), False
# SCAFFOLD:         except ... as exc:   # Exception
# SCAFFOLD:             output, failed = {"error": f"{type(exc).__name__}: {exc}"}, True
# SCAFFOLD:         print(f"  run {block.name}({block.input}) -> {output}")
# SCAFFOLD:         results.append({"type": "tool_result", "tool_use_id": ...,   # block.id
# SCAFFOLD:                         "content": json.dumps(output), "is_error": ...})   # failed
# SCAFFOLD:     return results
# END SOLUTION: replace each ... below with the value named beside it

check("safe_run_tools", run_tools=run_tools)

---

## Task 3 - Stop on stop_reason, cap the turns

In [ ]:
# ============================================================
# TASK 3 - Stop on stop_reason, cap the turns
# ============================================================
#
# WHAT TO DO
#   Write the harness: a loop that sends the conversation,
#   records each turn in a trace, stops when stop_reason is not
#   "tool_use" - and stops anyway after max_turns, saying so.
#   Run it normally, then with max_turns=1.
#
# WHY IT MATTERS
#   Claude decides which tool to call and in what order; your
#   loop runs them and decides when to stop - by reading
#   stop_reason, the only reliable signal. A turn cap is a
#   safety net for the day something loops, not the way a
#   normal run ends; when it fires, the result must say so.
#   The trace - turn, stop reason, tools - is what you read when
#   a run goes wrong.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Our loop ran them, and our loop decided when to stop".
#
# HOW TO DO IT
#   Three blanks:
#
#       max_turns + 1   the last turn number the loop may run
#       "tool_use"      anything else means Claude is done
#       "max_turns"     the status when the cap stopped the run
# ============================================================

try:
    # BEGIN SOLUTION
    def run_agent(text, max_turns=10):
        messages = [{"role": "user", "content": text}]
        trace = []
        for turn in range(1, max_turns + 1):
            response = client.messages.create(
                model=MODEL, max_tokens=500, output_config={"effort": "low"},
                system=SYSTEM, tools=TOOLS, messages=messages)
            trace.append({"turn": turn, "stop_reason": response.stop_reason,
                          "tools": [b.name for b in response.content if b.type == "tool_use"]})
            if response.stop_reason != "tool_use":
                text_out = next(b.text for b in response.content if b.type == "text")
                return {"status": "done", "text": text_out, "trace": trace}
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": run_tools(response)})
        return {"status": "max_turns", "text": None, "trace": trace}

    result = run_agent(QUESTION)
    capped = run_agent(QUESTION, max_turns=1)
    # SCAFFOLD: def run_agent(text, max_turns=10):
    # SCAFFOLD:     messages = [{"role": "user", "content": text}]
    # SCAFFOLD:     trace = []
    # SCAFFOLD:     for turn in range(1, ...):   # max_turns + 1
    # SCAFFOLD:         response = client.messages.create(
    # SCAFFOLD:             model=MODEL, max_tokens=500, output_config={"effort": "low"},
    # SCAFFOLD:             system=SYSTEM, tools=TOOLS, messages=messages)
    # SCAFFOLD:         trace.append({"turn": turn, "stop_reason": response.stop_reason,
    # SCAFFOLD:                       "tools": [b.name for b in response.content if b.type == "tool_use"]})
    # SCAFFOLD:         if response.stop_reason != ...:   # "tool_use"
    # SCAFFOLD:             text_out = next(b.text for b in response.content if b.type == "text")
    # SCAFFOLD:             return {"status": "done", "text": text_out, "trace": trace}
    # SCAFFOLD:         messages.append({"role": "assistant", "content": response.content})
    # SCAFFOLD:         messages.append({"role": "user", "content": run_tools(response)})
    # SCAFFOLD:     return {"status": ..., "text": None, "trace": trace}   # "max_turns"
    # SCAFFOLD:
    # SCAFFOLD: result = run_agent(QUESTION)
    # SCAFFOLD: capped = run_agent(QUESTION, max_turns=1)
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for run in (result, capped):
    print(run["status"], "-", run["text"])
    for entry in run["trace"]:
        print("   turn", entry["turn"], entry["stop_reason"], entry["tools"])

check("bounded_harness", result=result, capped=capped)

---

## Done

Send the conversation with the tools, read the stop reason, run every tool_use block and return every result with its id - failures included - and repeat until the stop reason is end_turn. Keep a trace of every turn, and a turn cap that says when it fired.